# 10 — Version 2 behavioural data preparation

**Owners:** Entire team  
**Run once before notebooks 11–15.**

This creates new files under `data/processed/v2`. It does not modify the
Version 1 Parquet files or notebooks.


## What Version 2 changes—and why

Version 1 remains our reproducible baseline. Version 2 adds behaviour that the
winning Kaggle solution showed was valuable, but implements it in a stricter
real-time form:

- `D` values are normalized against transaction day to expose stable date anchors.
- a conservative `uid_proxy` describes a possible client without using it as a label;
- counts, time since previous use, amount history and unique-value history describe
  behaviour;
- every historical feature uses only earlier transactions; and
- no feature reads `isFraud`, later rows, validation labels, or test labels.

The newest 15% remains the final test period. It is never used for feature or
hyperparameter selection.


In [ ]:
from pathlib import Path
_start = Path.cwd().resolve()
for _candidate in [_start, *_start.parents]:
    if (_candidate / "requirements-training.txt").exists():
        _requirements = _candidate / "requirements-training.txt"
        break
else:
    raise FileNotFoundError("Open this notebook from inside the cloned repository")
%pip install -q -r {_requirements}


In [ ]:
from pathlib import Path
import gc, json, os, sys, time
import numpy as np
import pandas as pd

def locate_project_root(start=None):
    candidate = Path(start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / ".git").exists() and (path / "src").exists():
            return path
    raise FileNotFoundError("Run this notebook from inside the cloned repository")

PROJECT_ROOT = locate_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

V2_DATA_DIR = PROJECT_ROOT / "data" / "processed" / "v2"
V2_ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "v2"
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("Project root:", PROJECT_ROOT)
print("Version 2 data:", V2_DATA_DIR)
print("Version 2 artifacts:", V2_ARTIFACT_ROOT)


## 1. Locate or download the labelled files

Existing CSVs or the existing competition ZIP are reused. If neither exists,
set `KAGGLE_API_TOKEN` in the machine's environment. Accept the competition
rules on Kaggle before running.


In [ ]:
import subprocess, zipfile

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "ieee-fraud-detection"
RAW_DIR.mkdir(parents=True, exist_ok=True)

def discover(name):
    matches = list(RAW_DIR.rglob(name))
    return matches[0] if matches else None

transaction_path = discover("train_transaction.csv")
identity_path = discover("train_identity.csv")
archive = RAW_DIR / "ieee-fraud-detection.zip"

if transaction_path is None or identity_path is None:
    if archive.exists():
        with zipfile.ZipFile(archive) as source:
            source.extractall(RAW_DIR)
    elif os.getenv("KAGGLE_API_TOKEN"):
        subprocess.run(
            ["kaggle", "competitions", "download", "-c", "ieee-fraud-detection", "-p", str(RAW_DIR)],
            check=True,
        )
        with zipfile.ZipFile(archive) as source:
            source.extractall(RAW_DIR)
    else:
        raise RuntimeError(
            "Place ieee-fraud-detection.zip in data/raw/ieee-fraud-detection or set KAGGLE_API_TOKEN."
        )
    transaction_path = discover("train_transaction.csv")
    identity_path = discover("train_identity.csv")

if transaction_path is None or identity_path is None:
    raise FileNotFoundError("The archive did not contain both labelled training CSV files")
print(transaction_path)
print(identity_path)


## 2. Load and left join

The transaction table defines the population. A left join preserves transactions
without identity/device records; missing identity is itself a useful signal.
`validate="one_to_one"` prevents accidental row multiplication.


In [ ]:
from src.fraud_pipeline.common import reduce_memory_usage

transactions = reduce_memory_usage(pd.read_csv(transaction_path))
identity = reduce_memory_usage(pd.read_csv(identity_path))
if not transactions["TransactionID"].is_unique:
    raise ValueError("Transaction IDs must be unique in train_transaction")
if not identity["TransactionID"].is_unique:
    raise ValueError("Transaction IDs must be unique in train_identity")

raw_columns = list(transactions.columns) + [c for c in identity.columns if c != "TransactionID"]
joined = transactions.merge(identity, on="TransactionID", how="left", validate="one_to_one")
if len(joined) != len(transactions):
    raise AssertionError("Left join changed the transaction row count")
print("Joined shape:", joined.shape)
del transactions, identity
gc.collect()


## 3. Generate strictly historical features

The dataframe is sorted by `TransactionDT` and `TransactionID`. Cumulative
statistics are shifted so the current row is excluded. Validation and test rows
may use attributes from earlier observed transactions, exactly as an online
transaction stream would, but never their labels.


In [ ]:
import joblib
from src.fraud_pipeline.behavioral import (
    BehavioralFeatureContract,
    add_causal_behavioral_features,
    build_behavioral_reference,
)
from src.fraud_pipeline.common import build_feature_audit, chronological_split
from src.fraud_pipeline.artifacts import write_json

raw_input_schema = {
    "required_columns": [c for c in raw_columns if c != "isFraud"],
    "target": "isFraud",
    "join": {"type": "left", "key": "TransactionID"},
}
ordered_features = add_causal_behavioral_features(joined, copy=True)
if len(ordered_features) != len(joined):
    raise AssertionError("Feature engineering changed the row count")
if ordered_features["TransactionID"].duplicated().any():
    raise AssertionError("Feature engineering duplicated a transaction")

# Keep the interactive leakage audit small enough for a 16 GB laptop.
# The repository unit test checks the same invariants independently.
leakage_sample = joined.nsmallest(
    min(20_000, len(joined)), ["TransactionDT", "TransactionID"]
).copy()
deployment_reference = build_behavioral_reference(joined)
del joined
gc.collect()

train, validation, test, split_metadata = chronological_split(ordered_features)
print({name: part.shape for name, part in {
    "train": train, "validation": validation, "test": test}.items()})


## 4. Save Version 2 partitions and deployment reference

The reference contains no fraud labels. It summarizes all labelled history only
for transactions that arrive after this dataset. The held-out metrics below never
use this reference; they use the past-only features already generated row by row.


In [ ]:
V2_DATA_DIR.mkdir(parents=True, exist_ok=True)
train.to_parquet(V2_DATA_DIR / "train.parquet", index=False)
validation.to_parquet(V2_DATA_DIR / "validation.parquet", index=False)
test.to_parquet(V2_DATA_DIR / "test.parquet", index=False)
build_feature_audit(train).to_csv(V2_DATA_DIR / "feature_audit.csv", index=False)
write_json(V2_DATA_DIR / "split_metadata.json", split_metadata)
write_json(V2_DATA_DIR / "raw_input_schema.json", raw_input_schema)
write_json(V2_DATA_DIR / "behavioral_contract.json", BehavioralFeatureContract().to_dict())

joblib.dump(deployment_reference, V2_DATA_DIR / "behavioral_reference.joblib", compress=3)
write_json(V2_DATA_DIR / "data_summary.json", {
    "version": "2.0", "joined_rows": len(ordered_features),
    "joined_columns_before_v2": len(raw_columns),
    "columns_after_v2": len(ordered_features.columns),
    "new_columns": [c for c in ordered_features.columns if c not in raw_columns],
})
print("Saved Version 2 data to:", V2_DATA_DIR)


## 5. Leakage assertions

These checks make the interview claim testable: the first occurrence of a proxy
has zero prior count, and changing the target cannot change engineered features.


In [ ]:
sample_features = add_causal_behavioral_features(leakage_sample, copy=True)
feature_columns = [c for c in sample_features if c != "isFraud"]
changed_target = leakage_sample.copy()
changed_target["isFraud"] = 1 - changed_target["isFraud"]
changed_features = add_causal_behavioral_features(changed_target, copy=True)
pd.testing.assert_frame_equal(
    sample_features[feature_columns], changed_features[feature_columns], check_dtype=False
)
first_for_uid = ordered_features.groupby("uid_proxy", observed=True).head(1)
assert (first_for_uid["uid_proxy_prior_count"] == 0).all()
assert train["TransactionDT"].max() <= validation["TransactionDT"].min()
assert validation["TransactionDT"].max() <= test["TransactionDT"].min()
print("Leakage and ordering checks passed.")


## Output

The next notebooks read these frozen Version 2 partitions. Share this processed
folder only if teammates cannot run preparation themselves; never commit raw or
processed competition data to Git.
